# Regressão Linear, Ridge e Lasso

Comparação de modelos lineares com regularização **L2 (Ridge)** e **L1 (Lasso)** em um problema de regressão.

## Objetivos
- criar atributos polinomiais;
- treinar uma regressão linear;
- observar o efeito da regularização Ridge;
- observar o efeito da regularização Lasso;
- comparar a magnitude dos coeficientes;
- discutir seleção de atributos.

**Tecnologias:** Python, NumPy, Matplotlib e Scikit-learn.

## 1. Carregamento dos dados

O notebook original utilizava `load_boston`, que foi removido das versões atuais do scikit-learn. Para manter o tema do projeto e permitir execução com versões modernas, os dados são obtidos do OpenML.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso

boston = fetch_openml(name="boston", version=1, as_frame=False, parser="auto")
X_raw = boston.data.astype(float)
y = boston.target.astype(float)

print("Amostras:", X_raw.shape[0])
print("Atributos originais:", X_raw.shape[1])

## 2. Escala e atributos polinomiais

A normalização é aplicada antes da expansão polinomial. Com grau 2, são criadas combinações entre os atributos originais.

In [ ]:
X_scaled = MinMaxScaler().fit_transform(X_raw)
X = PolynomialFeatures(degree=2, include_bias=False).fit_transform(X_scaled)
print("Atributos após PolynomialFeatures:", X.shape[1])

## 3. Regressão Linear

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

lr = LinearRegression().fit(X_train, y_train)
print(f"R² treinamento: {lr.score(X_train, y_train):.3f}")
print(f"R² teste: {lr.score(X_test, y_test):.3f}")

## 4. Ridge — regularização L2

O parâmetro `alpha` controla a intensidade da penalização dos coeficientes.

In [ ]:
ridge_models = {}
for alpha in [0.1, 1, 10]:
    model = Ridge(alpha=alpha).fit(X_train, y_train)
    ridge_models[alpha] = model
    print(f"Ridge alpha={alpha}: R² treino={model.score(X_train,y_train):.3f} | R² teste={model.score(X_test,y_test):.3f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(lr.coef_, "o", label="Linear Regression")
for alpha, model in ridge_models.items():
    plt.plot(model.coef_, marker=".", label=f"Ridge α={alpha}")
plt.axhline(0, linewidth=1)
plt.xlabel("Índice do coeficiente")
plt.ylabel("Magnitude")
plt.title("Efeito da regularização Ridge nos coeficientes")
plt.legend()
plt.show()

## 5. Lasso — regularização L1

Além de reduzir coeficientes, Lasso pode zerar alguns deles, o que pode ser útil em seleção de atributos.

In [ ]:
lasso_models = {}
for alpha in [1, 0.01, 0.0001]:
    model = Lasso(alpha=alpha, max_iter=100000).fit(X_train, y_train)
    lasso_models[alpha] = model
    non_zero = np.count_nonzero(model.coef_)
    print(f"Lasso alpha={alpha}: R² treino={model.score(X_train,y_train):.3f} | R² teste={model.score(X_test,y_test):.3f} | coeficientes não-zero={non_zero}")

In [ ]:
plt.figure(figsize=(10, 6))
for alpha, model in lasso_models.items():
    plt.plot(model.coef_, marker=".", label=f"Lasso α={alpha}")
plt.axhline(0, linewidth=1)
plt.xlabel("Índice do coeficiente")
plt.ylabel("Magnitude")
plt.title("Efeito da regularização Lasso nos coeficientes")
plt.legend()
plt.show()

## Conclusão

O experimento mostra como diferentes forças de regularização modificam os coeficientes de um modelo linear. Ridge tende a reduzir suas magnitudes, enquanto Lasso pode também produzir coeficientes exatamente iguais a zero.

Os valores de R² devem ser interpretados junto com a separação entre treino e teste e com o contexto do dataset.